<a href="https://colab.research.google.com/github/Tor4narek/AudioClassification/blob/main/sounds.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sound classifier from tabular audio features

This notebook prepares a neural network for 4 sound classes: `gunshot`, `sirena`, `dog_bark`, `children`.

Data is loaded from `X_y_basic_readable.xlsx`, sheet `dataset_basic`. Training is disabled by default with `RUN_TRAINING = False`.


In [ ]:
try:
    import openpyxl  # noqa: F401
except ImportError:
    import subprocess
    import sys

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openpyxl"])


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))


In [ ]:
DATA_PATH = "/content/drive/MyDrive/ai/X_y_basic_readable.xlsx"
OUTPUT_DIR = "/content/drive/MyDrive/ai"
SHEET_NAME = "dataset_basic"
RUN_TRAINING = False

FEATURE_COLUMNS = [
    "rms_mean",
    "active_duration",
    "rms_peak_to_mean",
    "attack_time",
    "zcr_mean",
    "spectral_centroid_mean",
    "spectral_bandwidth_mean",
    "spectral_rolloff_mean",
]

CLASS_NAMES = ["gunshot", "sirena", "dog_bark", "children"]
SEED = 42
VALIDATION_SIZE = 0.2
BATCH_SIZE = 16
EPOCHS = 80


In [ ]:
df = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)

required_columns = FEATURE_COLUMNS + CLASS_NAMES
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

empty_counts = df[required_columns].isna().sum()
if empty_counts.any():
    raise ValueError(f"Missing values found:\n{empty_counts[empty_counts > 0]}")

class_row_sums = df[CLASS_NAMES].sum(axis=1)
invalid_rows = df.index[class_row_sums != 1].tolist()
if invalid_rows:
    raise ValueError(f"Each row must have exactly one active class. Bad row indexes: {invalid_rows[:20]}")

print("Dataset shape:", df.shape)
print("Class balance:")
display(df[CLASS_NAMES].sum().rename("count").to_frame())

df.head()


In [ ]:
X = df[FEATURE_COLUMNS].astype("float32").to_numpy()
y = df[CLASS_NAMES].astype("float32").to_numpy()
y_labels = np.argmax(y, axis=1)

X_train, X_val, y_train, y_val, labels_train, labels_val = train_test_split(
    X,
    y,
    y_labels,
    test_size=VALIDATION_SIZE,
    random_state=SEED,
    stratify=y_labels,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype("float32")
X_val_scaled = scaler.transform(X_val).astype("float32")

print("X_train:", X_train_scaled.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val_scaled.shape)
print("y_val:", y_val.shape)


In [ ]:
tf.keras.utils.set_random_seed(SEED)

model = keras.Sequential(
    [
        keras.Input(shape=(len(FEATURE_COLUMNS),), name="audio_features"),
        layers.Dense(64, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.25),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.15),
        layers.Dense(len(CLASS_NAMES), activation="softmax", name="sound_class"),
    ],
    name="sound_classifier",
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()


In [ ]:
history = None

if RUN_TRAINING:
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=12,
            restore_best_weights=True,
        )
    ]

    history = model.fit(
        X_train_scaled,
        y_train,
        validation_data=(X_val_scaled, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=1,
    )
else:
    print("Training is disabled: RUN_TRAINING = False")


In [ ]:
if history is not None:
    history_df = pd.DataFrame(history.history)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history_df["accuracy"], label="Train Accuracy")
    plt.plot(history_df["val_accuracy"], label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.title("Accuracy")

    plt.subplot(1, 2, 2)
    plt.plot(history_df["loss"], label="Train Loss")
    plt.plot(history_df["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.title("Loss")

    plt.tight_layout()
    plt.show()
else:
    print("Training charts will appear after model.fit runs.")


In [ ]:
if RUN_TRAINING:
    loss, accuracy = model.evaluate(X_val_scaled, y_val, verbose=0)
    print(f"Validation loss: {loss:.4f}")
    print(f"Validation accuracy: {accuracy:.4f}")

    probabilities = model.predict(X_val_scaled, verbose=0)
    y_pred = np.argmax(probabilities, axis=1)
    y_true = np.argmax(y_val, axis=1)

    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        cmap="Blues",
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.show()

    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))
else:
    print("Evaluation is skipped because RUN_TRAINING = False")


In [ ]:
def predict_sound(
    rms_mean,
    active_duration,
    rms_peak_to_mean,
    attack_time,
    zcr_mean,
    spectral_centroid_mean,
    spectral_bandwidth_mean,
    spectral_rolloff_mean,
):
    """Return predicted class and probabilities for 8 audio features."""
    features = np.array(
        [[
            rms_mean,
            active_duration,
            rms_peak_to_mean,
            attack_time,
            zcr_mean,
            spectral_centroid_mean,
            spectral_bandwidth_mean,
            spectral_rolloff_mean,
        ]],
        dtype="float32",
    )

    scaled_features = scaler.transform(features).astype("float32")
    probabilities = model.predict(scaled_features, verbose=0)[0]
    predicted_index = int(np.argmax(probabilities))

    return {
        "predicted_class": CLASS_NAMES[predicted_index],
        "probabilities": {
            class_name: float(probability)
            for class_name, probability in zip(CLASS_NAMES, probabilities)
        },
    }

# Example after training:
# predict_sound(0.05, 0.85, 4.75, 0.48, 0.05, 1427.12, 2159.52, 2870.10)


In [ ]:
if RUN_TRAINING:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    model_path = os.path.join(OUTPUT_DIR, "sound_classifier_model.keras")
    class_names_path = os.path.join(OUTPUT_DIR, "class_names.json")
    scaler_params_path = os.path.join(OUTPUT_DIR, "scaler_params.json")

    model.save(model_path)

    with open(class_names_path, "w", encoding="utf-8") as f:
        json.dump(CLASS_NAMES, f, ensure_ascii=False, indent=2)

    scaler_params = {
        "feature_columns": FEATURE_COLUMNS,
        "mean": scaler.mean_.tolist(),
        "scale": scaler.scale_.tolist(),
    }
    with open(scaler_params_path, "w", encoding="utf-8") as f:
        json.dump(scaler_params, f, ensure_ascii=False, indent=2)

    print("Model and metadata saved:")
    print(model_path)
    print(class_names_path)
    print(scaler_params_path)
else:
    print("Saving is skipped because RUN_TRAINING = False")
